# Fine-tune a customer support assistant on a free GPU

This notebook runs the whole pipeline from
[llm-finetune-support](https://github.com/MITRAKER/llm-finetune-support) on Kaggle's free GPU:

1. **Prepare the data**: clean, deduplicate and split the Bitext customer-support dataset.
2. **Baseline**: measure the base model (Qwen3-4B-Instruct-2507) with the same system prompt.
3. **Train**: QLoRA fine-tuning with Unsloth (about 3 hours on one T4).
4. **Evaluate**: measure the fine-tuned model and apply the quality gates. If a gate fails, the notebook stops before export and tells you why, so you can change the config and retrain.
5. **Export**: merge the adapter, convert to a 4-bit GGUF file and (optionally) upload it to the Hugging Face Hub.

Every step is a script in the repository, so the same commands work on any CUDA machine. `LEARNING.md` explains what each step does and why.

## Before you run it

1. **Settings > Accelerator: GPU T4 x2** (needs a phone-verified Kaggle account). The pipeline uses one of the two GPUs.
2. **Settings > Internet: on** (to install packages and download the model and data).
3. **Add-ons > Secrets** (optional, tick "attach" for each one):
   - `HF_TOKEN`: a Hugging Face token with *write* access, to upload the model.
   - `WANDB_API_KEY`: to watch the loss curves live on Weights & Biases.
   - `ANTHROPIC_API_KEY`: to add the Claude LLM-judge scores to the evaluation (costs a few dollars with the default model; see the config).
4. Set `HUB_MODEL_ID` in the settings cell below if you want to upload the model.
5. First do a quick test with `QUICK_TEST = True` (30 to 40 minutes, including the GGUF export). Then set it to `False` and use **Save Version > Save & Run All** so the full run (4 to 5 hours) continues in the background after you close the tab.

The run uses about 5 of your 30 free GPU hours per week. Outputs (metrics, report, adapter, GGUF) are saved in `/kaggle/working` with the notebook version.

## 0. Check the GPU

In [ ]:
# Do not import torch before the install cell: if the install upgrades it, the old version
# would stay loaded in this session.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!pip show torch 2>/dev/null | head -2

## 1. Settings

In [ ]:
QUICK_TEST = True   # True: 60 training steps and 50 test examples, to check every step in 30-40 minutes
HUB_MODEL_ID = ""   # e.g. "your-hf-username/qwen3-4b-support"; empty = keep the model in this notebook only
REPO_URL = "https://github.com/MITRAKER/llm-finetune-support.git"
REPO_REF = "main"   # a branch or a tag such as "v0.1.0" for an exactly reproducible run

## 2. Get the code and install the training stack

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
REPO_DIR = WORK_ROOT / "llm-finetune-support"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Code version:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# Pinned versions that work together (see requirements/train.txt). uv resolves and installs
# much faster than pip, and keeps the GPU image's own torch whenever it fits the pins.
if shutil.which("uv"):
    !uv pip install --system --quiet -r requirements/train.txt --torch-backend=auto
else:
    !pip install --quiet -r requirements/train.txt

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
major, minor = torch.cuda.get_device_capability(0)
print(f"Compute capability {major}.{minor}: training in {'bf16' if major >= 8 else 'fp16 (a T4 has no fast bf16)'}")

## 3. Secrets

Secrets are copied into environment variables, which is where the libraries look for them. Their values are never printed.

In [ ]:
SECRET_NAMES = ["HF_TOKEN", "WANDB_API_KEY", "ANTHROPIC_API_KEY"]


def load_secrets(names):
    loaded = []
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        get = UserSecretsClient().get_secret
    except ImportError:
        try:
            from google.colab import userdata  # Colab
            get = userdata.get
        except ImportError:
            return [n for n in names if os.environ.get(n)]  # local machine: already in the environment
    for name in names:
        try:
            value = get(name)
        except Exception:  # secret not added or not attached to this notebook
            continue
        if value:
            os.environ[name] = value
            loaded.append(name)
    return loaded


print("Secrets available:", load_secrets(SECRET_NAMES) or "none")
os.environ.setdefault("WANDB_PROJECT", "llm-finetune-support")

## 4. Run config

The experiment config lives in `configs/qwen3_4b_qlora.yaml`. This cell copies it and changes only what this run needs, so the original stays the single source of truth.

In [ ]:
import yaml

cfg = yaml.safe_load(Path("configs/qwen3_4b_qlora.yaml").read_text())
if QUICK_TEST:
    cfg["train"]["max_steps"] = 60
    cfg["eval"]["n_samples"] = 50
cfg["export"]["hub_model_id"] = HUB_MODEL_ID or None
# The 16-bit merge is about 8 GB: keep it on scratch disk, not in the saved notebook output.
cfg["export"]["merged_dir"] = "/tmp/merged"
Path("configs/run.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
CONFIG = "configs/run.yaml"


def run(*args):
    # Run one pipeline step and return its exit code (a failing step does not stop the cell).
    result = subprocess.run([sys.executable, *args])
    print(f"exit code {result.returncode}")
    return result.returncode


print(yaml.safe_dump({k: cfg[k] for k in ("model", "lora", "train")}, sort_keys=False))

## 5. Step 1: prepare the data

In [ ]:
assert run("src/prepare_data.py", "--config", CONFIG) == 0

## 6. Step 2: baseline

The base model gets exactly the same system prompt the fine-tuned model will get, including the placeholder instruction. That makes the comparison fair: any improvement comes from the training, not from a better prompt.

In [ ]:
assert run("src/evaluate.py", "--config", CONFIG, "--model", "base") == 0

## 7. Step 3: fine-tune

Watch `loss` (training) and `eval_loss` (validation). Both should fall quickly and then flatten. If `eval_loss` starts rising while `loss` keeps falling, the model is memorising the training answers (overfitting). The best checkpoint by `eval_loss` is kept automatically.

In [ ]:
assert run("src/train.py", "--config", CONFIG) == 0

## 8. Step 4: evaluate the fine-tuned model and apply the quality gates

In [ ]:
from IPython.display import Markdown, display

assert run("src/evaluate.py", "--config", CONFIG, "--model", "tuned") == 0
gates_exit_code = run("src/evaluate.py", "--config", CONFIG, "--compare")
display(Markdown(Path("results/REPORT.md").read_text()))
GATES_PASSED = gates_exit_code == 0
if not GATES_PASSED:
    print("A quality gate failed. Read results/tuned_predictions.jsonl to see which answers are wrong,")
    print("change the data or hyperparameters in the config, and train again before exporting.")

## 9. Step 5: export (and upload)

Merges the adapter into the base model, converts it to GGUF and quantizes it to 4 bits (about 2.5 GB), so it runs on a CPU. With `HUB_MODEL_ID` and `HF_TOKEN` set, it also uploads the adapter and the GGUF file with a model card that includes the evaluation report.

In [ ]:
if GATES_PASSED or QUICK_TEST:
    push = ["--push"] if HUB_MODEL_ID and os.environ.get("HF_TOKEN") and not QUICK_TEST else []
    assert run("src/export.py", "--config", CONFIG, *push) == 0
    for gguf in Path("/tmp").glob("merged*/**/*.gguf"):
        if "Q4_K_M" in gguf.name.upper():
            shutil.copy(gguf, WORK_ROOT / gguf.name)  # keep it with the notebook output
            print("Saved", WORK_ROOT / gguf.name)
else:
    print("Skipped: the quality gates failed (see the report above).")

## 10. Keep the results

Everything in `/kaggle/working` is saved with this notebook version: `results/` (metrics, predictions, `REPORT.md`), `outputs/` (the adapter and `training_summary.json`) and the GGUF file.

Next steps:

- Copy the comparison table from `results/REPORT.md` into the README's results section.
- Deploy the demo: in the GitHub repository set the `HF_TOKEN` secret and the `HF_SPACE_ID` and `HF_MODEL_REPO` variables (`HF_MODEL_REPO` is `HUB_MODEL_ID` + `-GGUF`). The deploy workflow then publishes `app/` to your Space.